# Load data

In [ ]:
import pandas as pd  

df = pd.read_csv("Salary_Data.csv")
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
# Education Level etiketlerini tekilleştir (Bachelor's / Bachelor's Degree gibi tekrarlar var)
edu_map = {
    "Bachelor's Degree": "Bachelor's",
    "Master's Degree": "Master's",
    "phD": "PhD",
}
df["Education Level"] = df["Education Level"].replace(edu_map)

# Özellik mühendisliği: meslek unvanından kıdem bilgisi (Job Title string iken yapılmalı)
title = df["Job Title"].str.lower()
df["Is_Senior"]  = title.str.contains("senior|director|chief|vp|head").astype(int)
df["Is_Junior"]  = title.str.contains("junior|entry|intern").astype(int)
df["Is_Manager"] = title.str.contains("manager|director|chief|vp|head").astype(int)

# İşe başlama yaşı
df["Start_Age"] = df["Age"] - df["Years of Experience"]
df[["Job Title", "Age", "Years of Experience", "Start_Age", "Is_Senior", "Is_Junior", "Is_Manager"]].head()

In [ ]:
col = "Job Title"
threshold = 5  # bu sayıdan az geçen sınıflar "rare"

counts = df[col].value_counts()
rare = counts[counts < threshold]

# sınıf adları object olarak
rare_classes = rare.index.astype(object)
print(rare_classes)
print(type(rare_classes[0]))  # <class 'str'>

# sınıf adı + kaç kez geçtiği
rare_df = rare.rename_axis(col).reset_index(name="count")
rare_df


In [ ]:
rare_df.describe()

In [ ]:
# Sadece korelasyon/heatmap için kodlanmış KOPYA. Model bu kopyayı kullanmaz.
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()
for c in ["Gender", "Education Level", "Job Title"]:
    df_enc[c] = LabelEncoder().fit_transform(df_enc[c])
df_enc.head()

In [ ]:
df_enc.corr()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.heatmap(df_enc.corr(), annot=True, cmap="RdYlGn")
plt.show()





In [ ]:
# scatter plot
sns.scatterplot(x="Age", y="Salary", data=df)
plt.show()

In [ ]:
# violin plot gender salary
sns.violinplot(x="Gender", y="Salary", data=df)
plt.show()

In [ ]:
# outlier detection with iqr threshold
q1 = df["Salary"].quantile(0.25)
q3 = df["Salary"].quantile(0.75)
iqr = q3 - q1
threshold = 1.5 * iqr
lower_bound = q1 - threshold
upper_bound = q3 + threshold



In [ ]:
#remove outliers
df = df[df["Salary"] > lower_bound]
df = df[df["Salary"] < upper_bound]

# histogram
sns.histplot(df["Salary"], bins=20)
plt.show()


In [ ]:
df.head()



## Modelleme
`Age` sütunu artık silinmiyor. Veri önce `train_test_split` ile bölünüyor, ardından preprocessing pipeline içinde yapılıyor.

In [ ]:
from sklearn.model_selection import train_test_split

# Age sütunu SİLİNMİYOR. Veri önce bölünüyor, scaling/encoding bölmeden sonra yapılıyor.
X = df.drop(columns=["Salary"])
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["Age", "Years of Experience", "Start_Age"]
bin_cols = ["Is_Senior", "Is_Junior", "Is_Manager"]
cat_cols = ["Gender", "Education Level", "Job Title"]

# ColumnTransformer Pipeline içinde fit edildiği için scaler/encoder sadece X_train'i görür (data leakage yok)
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

models = {
    "LinearRegression (baseline)": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
}

fitted, scores = {}, {}
for name, m in models.items():
    pipe = Pipeline([("pre", preprocessor), ("model", m)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    scores[name] = r2_score(y_test, pipe.predict(X_test))
    print(f"{name:30s} R2 = {scores[name]:.4f}")

In [ ]:
best_name = max(scores, key=scores.get)
model = fitted[best_name]
y_pred = model.predict(X_test)
print("En iyi model:", best_name)
y_pred

In [ ]:
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
mse

In [ ]:
mae

In [ ]:
r2